# KCGS 환경등급과 GIR·DART 환경성과 재검증

## 분석 정렬

- **PRIMARY:** KCGS 평가년도 (t) → GIR 대상년도 (t-1) → DART 재무연도 (t-1)
- **SENSITIVITY:** KCGS 평가년도 (t) → GIR 대상년도 (t) → DART 재무연도 (t)

KCGS 당해 평가가 전년도 평가대상기간과 연결되는 구조를 고려해 1년 시차 정렬을 주 분석으로 사용한다. 동일연도 정렬은 기존 분석을 재현하는 민감도 분석으로 유지한다.

KCGS 환경등급은 온실가스만으로 결정되는 지표가 아니며, (t-1) 정렬은 인과관계를 뜻하지 않는다. 환경등급 숫자는 순서형 분석을 위한 값일 뿐 등급 간 거리가 같다는 뜻도 아니다.

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr
from IPython.display import display

data_path = Path("../data")
grade_order = ["A+", "A", "B+", "B", "C", "D"]
grade_rank = {"D": 1, "C": 2, "B": 3, "B+": 4, "A": 5, "A+": 6}

def normalize_company_name(name):
    name = str(name).strip()
    for word in ["주식회사", "(주)", "㈜", "유한회사", "(유)"]:
        name = name.replace(word, "")
    return re.sub(r"\s+", "", name)

def clean_kcgs_code(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    text = re.sub(r"\.0$", "", text)
    if text.isdigit() and len(text) <= 6:
        return text.zfill(6)
    return None

def clean_dart_code(value):
    text = str(value).strip()
    bracketed = re.fullmatch(r"\[(\d{6})\]", text)
    if bracketed:
        return bracketed.group(1)
    if re.fullmatch(r"\d{6}", text):
        return text
    return None

kcgs = pd.read_csv(data_path / "kcgs.csv")
gir = pd.read_csv(data_path / "gir_company_2020_2025_reviewed.csv")

kcgs = kcgs[kcgs["평가년도"].between(2020, 2025)].copy()
gir = gir[gir["대상년도"].between(2020, 2025)].copy()
kcgs["평가년도"] = kcgs["평가년도"].astype(int)
gir["대상년도"] = gir["대상년도"].astype(int)

kcgs["기업명_clean"] = kcgs["기업명"].astype(str).str.strip()
gir["기업명_clean"] = gir["법인명"].astype(str).str.strip()
kcgs["기업명_norm"] = kcgs["기업명_clean"].map(normalize_company_name)
gir["기업명_norm"] = gir["기업명_clean"].map(normalize_company_name)
kcgs["종목코드_clean"] = kcgs["기업코드"].map(clean_kcgs_code)

kcgs_raw_dup = kcgs.duplicated(["기업명_clean", "평가년도"]).sum()
gir_raw_dup = gir.duplicated(["기업명_clean", "대상년도"]).sum()
kcgs_norm_dup = kcgs.duplicated(["기업명_norm", "평가년도"]).sum()
gir_norm_dup = gir.duplicated(["기업명_norm", "대상년도"]).sum()

kcgs_collision = (
    kcgs.groupby(["기업명_norm", "평가년도"])["기업명_clean"].nunique().gt(1).sum()
)
gir_collision = (
    gir.groupby(["기업명_norm", "대상년도"])["기업명_clean"].nunique().gt(1).sum()
)

print("KCGS 기업명+연도 중복:", int(kcgs_raw_dup))
print("GIR 기업명+연도 중복:", int(gir_raw_dup))
print("KCGS 정규화명+연도 중복:", int(kcgs_norm_dup))
print("GIR 정규화명+연도 중복:", int(gir_norm_dup))
print("KCGS 서로 다른 원명이 같은 정규화명으로 충돌한 키:", int(kcgs_collision))
print("GIR 서로 다른 원명이 같은 정규화명으로 충돌한 키:", int(gir_collision))
print("KCGS 기업연도수:", len(kcgs), "고유기업수:", kcgs["기업명_norm"].nunique())
print("GIR 기업연도수:", len(gir), "고유기업수:", gir["기업명_norm"].nunique())

assert kcgs_norm_dup == 0 and gir_norm_dup == 0
assert kcgs_collision == 0 and gir_collision == 0

KCGS 기업명+연도 중복: 0
GIR 기업명+연도 중복: 0
KCGS 정규화명+연도 중복: 0
GIR 정규화명+연도 중복: 0
KCGS 서로 다른 원명이 같은 정규화명으로 충돌한 키: 0
GIR 서로 다른 원명이 같은 정규화명으로 충돌한 키: 0
KCGS 기업연도수: 6203 고유기업수: 1391
GIR 기업연도수: 6710 고유기업수: 1505


## DART 매출액 처리

DART 종목코드는 `[123456]` 또는 `123456` 형태의 정확한 숫자 6자리만 허용한다. 영문이 포함된 코드는 숫자로 강제 변환하지 않는다. 매출액은 `ifrs-full_Revenue`만 사용하고, 기업·연도·별도/연결별 최신 결산일을 선택한다.

최신 결산일에서 서로 다른 Revenue가 충돌하면 임의 선택하지 않고 분석을 중단한다. 환경성과 분석에서는 GIR의 법인 단위 배출경계에 가깝게 맞추기 위해 별도재무제표를 우선하고 연결재무제표를 차순위로 사용한다. 다만 GIR 법인 경계와 DART 재무제표 경계가 완전히 일치하는 것은 아니다.

In [2]:
dart_revenue_list = []

for year in range(2020, 2026):
    df = pd.read_csv(
        data_path / f"dart_pl_{year}.csv",
        usecols=["재무제표종류", "종목코드", "회사명", "결산기준일", "통화", "항목코드", "항목명", "당기"],
        low_memory=False,
    )
    df["종목코드_clean"] = df["종목코드"].map(clean_dart_code)
    df = df[
        df["종목코드_clean"].notna()
        & (df["항목코드"] == "ifrs-full_Revenue")
    ].copy()

    df["매출액"] = pd.to_numeric(
        df["당기"].astype(str).str.replace(",", "", regex=False),
        errors="coerce",
    )
    df["결산기준일_clean"] = pd.to_datetime(df["결산기준일"], errors="coerce")
    df["연결여부"] = np.where(
        df["재무제표종류"].astype(str).str.contains("연결", na=False),
        "연결",
        "별도",
    )
    df["재무연도"] = year

    latest_date = (
        df.groupby(["종목코드_clean", "재무연도", "연결여부"])["결산기준일_clean"]
        .transform("max")
    )
    df = df[df["결산기준일_clean"].eq(latest_date)].copy()
    dart_revenue_list.append(df)

dart_latest = pd.concat(dart_revenue_list, ignore_index=True)
conflict_keys = (
    dart_latest.groupby(["종목코드_clean", "재무연도", "연결여부"])["매출액"]
    .nunique(dropna=True)
)
conflicting_revenue_count = int(conflict_keys.gt(1).sum())
print("conflicting_revenue_count ==", conflicting_revenue_count)
assert conflicting_revenue_count == 0, "최신 결산일의 Revenue 충돌을 확인해야 합니다."

dart_latest = dart_latest.drop_duplicates(
    ["종목코드_clean", "재무연도", "연결여부", "매출액"]
).copy()
dart_latest["우선순위"] = dart_latest["연결여부"].map({"별도": 0, "연결": 1})
dart_revenue_final = (
    dart_latest.sort_values(["종목코드_clean", "재무연도", "우선순위"])
    .drop_duplicates(["종목코드_clean", "재무연도"], keep="first")
    .copy()
)

print("DART 최종 기업연도수:", len(dart_revenue_final))
display(dart_revenue_final["연결여부"].value_counts().rename("기업연도수").to_frame())

conflicting_revenue_count == 0
DART 최종 기업연도수: 14381


,기업연도수
연결여부,
별도,14156
연결,225


In [3]:
def safe_spearman(df, x, y):
    temp = df[[x, y]].dropna()
    if len(temp) < 2 or temp[x].nunique() < 2 or temp[y].nunique() < 2:
        return np.nan, np.nan, len(temp)
    rho, p_value = spearmanr(temp[x], temp[y])
    return float(rho), float(p_value), len(temp)

def build_aligned(offset, label):
    left = kcgs.copy()
    left["성과연도"] = left["평가년도"] + offset

    kg = pd.merge(
        left,
        gir,
        left_on=["기업명_norm", "성과연도"],
        right_on=["기업명_norm", "대상년도"],
        how="inner",
        suffixes=("_kcgs", "_gir"),
        validate="one_to_one",
    )
    kg = kg[kg["환경"].isin(grade_order)].copy()
    kg["환경등급순위"] = kg["환경"].map(grade_rank)
    kg["기업키"] = np.where(
        kg["종목코드_clean"].notna(),
        "C:" + kg["종목코드_clean"].fillna(""),
        "N:" + kg["기업명_norm"],
    )

    combined = pd.merge(
        kg,
        dart_revenue_final[
            ["종목코드_clean", "재무연도", "매출액", "연결여부", "결산기준일_clean", "통화"]
        ],
        left_on=["종목코드_clean", "성과연도"],
        right_on=["종목코드_clean", "재무연도"],
        how="left",
        validate="many_to_one",
    )
    combined["배출집약도"] = np.nan
    valid_intensity = (
        combined["통화"].eq("KRW")
        & combined["매출액"].gt(0)
        & combined["온실가스 배출량(tCO₂eq)"].notna()
    )
    combined.loc[valid_intensity, "배출집약도"] = (
        combined.loc[valid_intensity, "온실가스 배출량(tCO₂eq)"]
        / (combined.loc[valid_intensity, "매출액"] / 1_000_000_000)
    )
    combined["정렬"] = label
    return combined

primary = build_aligned(-1, "PRIMARY(t-1)")
same_year = build_aligned(0, "SENSITIVITY(same-year)")

for label, frame in [("PRIMARY(t-1)", primary), ("SENSITIVITY(same-year)", same_year)]:
    print(label)
    print("기업연도수:", len(frame), "고유기업수:", frame["기업키"].nunique())
    print("배출집약도 유효 기업연도수:", frame["배출집약도"].notna().sum(),
          "고유기업수:", frame.loc[frame["배출집약도"].notna(), "기업키"].nunique())

PRIMARY(t-1)
기업연도수: 1149 고유기업수: 270
배출집약도 유효 기업연도수: 1141 고유기업수: 270
SENSITIVITY(same-year)
기업연도수: 1375 고유기업수: 279
배출집약도 유효 기업연도수: 1366 고유기업수: 279


## PRIMARY(t-1) 핵심 결과

절대배출량과 매출 10억원당 배출집약도를 구분한다. Spearman p-value도 계산하지만, 같은 기업이 여러 연도 반복되는 패널 자료이므로 관측치가 완전히 독립적이지 않다. 상관관계는 인과관계를 뜻하지 않는다.

In [4]:
def grade_summary(frame, value, mean_name, median_name):
    valid = frame.dropna(subset=[value]).copy()
    return (
        valid.groupby("환경")
        .agg(
            기업연도수=("기업키", "size"),
            고유기업수=("기업키", "nunique"),
            **{
                mean_name: (value, "mean"),
                median_name: (value, "median"),
            },
        )
        .reindex(grade_order)
    )

primary_ghg_summary = grade_summary(
    primary, "온실가스 배출량(tCO₂eq)", "평균배출량", "중앙값배출량"
)
primary_intensity_summary = grade_summary(
    primary, "배출집약도", "평균집약도", "중앙값집약도"
)

primary_ghg_rho, primary_ghg_p, primary_ghg_n = safe_spearman(
    primary, "환경등급순위", "온실가스 배출량(tCO₂eq)"
)
primary_int_rho, primary_int_p, primary_int_n = safe_spearman(
    primary, "환경등급순위", "배출집약도"
)

def yearly_correlations(frame, value):
    rows = []
    for year, group in frame.groupby("평가년도"):
        rho, p_value, n = safe_spearman(group, "환경등급순위", value)
        rows.append({"평가년도": year, "기업연도수": n, "Spearman": rho, "p-value": p_value})
    return pd.DataFrame(rows).set_index("평가년도")

primary_ghg_year_corr = yearly_correlations(primary, "온실가스 배출량(tCO₂eq)")
primary_int_year_corr = yearly_correlations(primary, "배출집약도")

print("환경등급별 절대 온실가스 배출량")
display(primary_ghg_summary.round(1))
print("환경등급별 배출집약도 (tCO₂eq / 매출 10억원)")
display(primary_intensity_summary.round(1))
print(f"환경등급 vs 절대배출량 전체 Spearman: rho={primary_ghg_rho:.4f}, p={primary_ghg_p:.4g}, n={primary_ghg_n}")
display(primary_ghg_year_corr.round(4))
print(f"환경등급 vs 배출집약도 전체 Spearman: rho={primary_int_rho:.4f}, p={primary_int_p:.4g}, n={primary_int_n}")
display(primary_int_year_corr.round(4))

환경등급별 절대 온실가스 배출량


,기업연도수,고유기업수,평균배출량,중앙값배출량
환경,,,,
A+,41,27,1047547.0,162990.0
A,302,114,948420.0,146235.0
B+,239,129,1021544.1,89065.0
B,161,123,389596.9,82004.0
C,357,137,120887.1,51887.0
D,49,45,114763.9,34404.0


환경등급별 배출집약도 (tCO₂eq / 매출 10억원)


,기업연도수,고유기업수,평균집약도,중앙값집약도
환경,,,,
A+,41,27,95.3,50.8
A,299,114,368.1,63.5
B+,238,128,520.2,89.1
B,161,123,514.4,136.6
C,353,137,391.1,172.3
D,49,45,326.2,186.5


환경등급 vs 절대배출량 전체 Spearman: rho=0.3453, p=1.592e-33, n=1149


,기업연도수,Spearman,p-value
평가년도,,,
2021,222,0.3168,0.0
2022,227,0.4288,0.0
2023,228,0.3705,0.0
2024,233,0.3620,0.0
2025,239,0.3591,0.0


환경등급 vs 배출집약도 전체 Spearman: rho=-0.3341, p=3.683e-31, n=1141


,기업연도수,Spearman,p-value
평가년도,,,
2021,222,-0.3388,0.0
2022,227,-0.3204,0.0
2023,228,-0.3125,0.0
2024,229,-0.3390,0.0
2025,235,-0.3518,0.0


In [5]:
def make_changes(frame):
    change = frame.sort_values(["기업키", "평가년도"]).copy()
    grouped = change.groupby("기업키", sort=False)

    change["전년도"] = grouped["평가년도"].shift(1)
    change["전년환경등급순위"] = grouped["환경등급순위"].shift(1)
    change["전년배출량"] = grouped["온실가스 배출량(tCO₂eq)"].shift(1)
    change["전년배출집약도"] = grouped["배출집약도"].shift(1)

    change = change[change["평가년도"].sub(change["전년도"]).eq(1)].copy()
    change["등급변화"] = change["환경등급순위"] - change["전년환경등급순위"]
    change["등급변화구분"] = np.select(
        [change["등급변화"].gt(0), change["등급변화"].lt(0)],
        ["상향", "하향"],
        default="유지",
    )

    change["배출량증감률"] = np.where(
        change["전년배출량"].gt(0),
        (change["온실가스 배출량(tCO₂eq)"] - change["전년배출량"])
        / change["전년배출량"] * 100,
        np.nan,
    )
    change["집약도증감률"] = np.where(
        change["전년배출집약도"].gt(0),
        (change["배출집약도"] - change["전년배출집약도"])
        / change["전년배출집약도"] * 100,
        np.nan,
    )
    change["비교기간"] = (
        change["전년도"].astype(int).astype(str)
        + "→"
        + change["평가년도"].astype(int).astype(str)
    )
    return change

def change_summary(change):
    rows = []
    for category in ["상향", "유지", "하향"]:
        group = change[change["등급변화구분"].eq(category)]
        valid = group.dropna(subset=["배출량증감률", "집약도증감률"])
        rows.append({
            "등급변화구분": category,
            "기업연도수": len(group),
            "유효비교수": len(valid),
            "고유기업수": valid["기업키"].nunique(),
            "배출량증감률_중앙값": valid["배출량증감률"].median(),
            "배출량감소기업수": valid["배출량증감률"].lt(0).sum(),
            "배출량감소비율": valid["배출량증감률"].lt(0).mean() * 100 if len(valid) else np.nan,
            "집약도증감률_중앙값": valid["집약도증감률"].median(),
            "집약도감소기업수": valid["집약도증감률"].lt(0).sum(),
            "집약도감소비율": valid["집약도증감률"].lt(0).mean() * 100 if len(valid) else np.nan,
        })
    return pd.DataFrame(rows).set_index("등급변화구분")

def intensity_recent_summary(change):
    valid = change.dropna(subset=["집약도증감률"]).copy()
    return (
        valid.groupby("등급변화구분")
        .agg(
            유효비교수=("기업키", "size"),
            고유기업수=("기업키", "nunique"),
            집약도증감률_중앙값=("집약도증감률", "median"),
            집약도감소기업수=("집약도증감률", lambda x: x.lt(0).sum()),
        )
        .assign(집약도감소비율=lambda x: x["집약도감소기업수"] / x["유효비교수"] * 100)
        .reindex(["상향", "유지", "하향"])
    )

primary_change = make_changes(primary)
primary_change_summary = change_summary(primary_change)

quantiles = pd.DataFrame({
    "1%": primary_change[["배출량증감률", "집약도증감률"]].quantile(0.01),
    "99%": primary_change[["배출량증감률", "집약도증감률"]].quantile(0.99),
    "유효비교수": primary_change[["배출량증감률", "집약도증감률"]].count(),
})

recent_primary = primary_change[primary_change["평가년도"].isin([2024, 2025])].copy()
recent_primary_by_period = pd.concat(
    {
        period: intensity_recent_summary(group)
        for period, group in recent_primary.groupby("비교기간")
    },
    names=["비교기간", "등급변화구분"],
)
recent_primary_combined = intensity_recent_summary(recent_primary)
recent_primary_rho, recent_primary_p, recent_primary_n = safe_spearman(
    recent_primary, "등급변화", "집약도증감률"
)

print("PRIMARY 동일기업 연속 평가연도 변화")
display(primary_change_summary.round(2))
print("증감률 1%, 99% 분위수 (극단값은 삭제하지 않음)")
display(quantiles.round(2))
print("최근 구간별: PRIMARY의 평가년도 변화는 성과연도 기준 한 해 앞선 변화와 연결")
display(recent_primary_by_period.round(2))
print("최근 두 구간 합산")
display(recent_primary_combined.round(2))
print(f"최근 등급 변화폭 vs 집약도 변화율 Spearman: rho={recent_primary_rho:.4f}, p={recent_primary_p:.6g}, n={recent_primary_n}")


PRIMARY 동일기업 연속 평가연도 변화


,기업연도수,유효비교수,고유기업수,배출량증감률_중앙값,배출량감소기업수,배출량감소비율,집약도증감률_중앙값,집약도감소기업수,집약도감소비율
등급변화구분,,,,,,,,,
상향,218,215,166,0.10,105,48.84,-13.14,177,82.33
유지,467,460,220,-0.34,236,51.30,-4.04,287,62.39
하향,190,189,169,1.42,84,44.44,-6.59,120,63.49


증감률 1%, 99% 분위수 (극단값은 삭제하지 않음)


,1%,99%,유효비교수
배출량증감률,-51.02,106.53,875
집약도증감률,-67.64,119.24,864


최근 구간별: PRIMARY의 평가년도 변화는 성과연도 기준 한 해 앞선 변화와 연결


유효비교수  고유기업수  집약도증감률_중앙값  집약도감소기업수  집약도감소비율
비교기간      등급변화구분                                             
2023→2024 상향         38     38       -6.14        27    71.05
          유지        144    144        1.23        64    44.44
          하향         37     37        9.64        11    29.73
2024→2025 상향         29     29       -4.98        20    68.97
          유지        177    177       -2.77       106    59.89
          하향         13     13        6.54         3    23.08

최근 두 구간 합산


,유효비교수,고유기업수,집약도증감률_중앙값,집약도감소기업수,집약도감소비율
등급변화구분,,,,,
상향,67,64,-5.17,47,70.15
유지,321,202,-1.23,170,52.96
하향,50,49,8.02,14,28.00


최근 등급 변화폭 vs 집약도 변화율 Spearman: rho=-0.2372, p=5.11663e-07, n=438


## SAME-YEAR 민감도 분석

동일연도 정렬은 기존 분석과의 비교를 위해 유지한다. 아래 표는 same-year와 PRIMARY(t-1)의 방향, 크기와 해석이 얼마나 달라지는지 보여준다.

In [6]:
same_ghg_rho, same_ghg_p, same_ghg_n = safe_spearman(
    same_year, "환경등급순위", "온실가스 배출량(tCO₂eq)"
)
same_int_rho, same_int_p, same_int_n = safe_spearman(
    same_year, "환경등급순위", "배출집약도"
)
same_change = make_changes(same_year)
recent_same = same_change[same_change["평가년도"].isin([2024, 2025])].copy()
recent_same_summary = intensity_recent_summary(recent_same)
recent_same_rho, recent_same_p, recent_same_n = safe_spearman(
    recent_same, "등급변화", "집약도증감률"
)

def direction_text(same_value, primary_value):
    if pd.isna(same_value) or pd.isna(primary_value):
        return "비교 불가"
    same_sign = np.sign(same_value)
    primary_sign = np.sign(primary_value)
    direction = "방향 동일" if same_sign == primary_sign else "방향 변화"
    magnitude = "t-1에서 절대값 증가" if abs(primary_value) > abs(same_value) else "t-1에서 절대값 감소"
    return f"{direction}; {magnitude}"

sensitivity_rows = [
    {
        "항목": "등급 vs 절대배출량 Spearman",
        "SAME-YEAR": same_ghg_rho,
        "t-1": primary_ghg_rho,
        "해석": direction_text(same_ghg_rho, primary_ghg_rho),
    },
    {
        "항목": "등급 vs 배출집약도 Spearman",
        "SAME-YEAR": same_int_rho,
        "t-1": primary_int_rho,
        "해석": direction_text(same_int_rho, primary_int_rho),
    },
    {
        "항목": "최근 등급변화폭 vs 집약도변화율 Spearman",
        "SAME-YEAR": recent_same_rho,
        "t-1": recent_primary_rho,
        "해석": direction_text(recent_same_rho, recent_primary_rho),
    },
]

for category in ["상향", "유지", "하향"]:
    same_value = recent_same_summary.loc[category, "집약도증감률_중앙값"]
    primary_value = recent_primary_combined.loc[category, "집약도증감률_중앙값"]
    sensitivity_rows.append({
        "항목": f"최근 {category} 집약도증감률 중앙값(%)",
        "SAME-YEAR": same_value,
        "t-1": primary_value,
        "해석": direction_text(same_value, primary_value),
    })

sensitivity_table = pd.DataFrame(sensitivity_rows).set_index("항목")
display(sensitivity_table.round(4))
print(f"SAME-YEAR 최근 변화 Spearman p-value={recent_same_p:.6g}, n={recent_same_n}")
print(f"PRIMARY(t-1) 최근 변화 Spearman p-value={recent_primary_p:.6g}, n={recent_primary_n}")

,SAME-YEAR,t-1,해석
항목,,,
등급 vs 절대배출량 Spearman,0.3466,0.3453,방향 동일; t-1에서 절대값 감소
등급 vs 배출집약도 Spearman,-0.3180,-0.3341,방향 동일; t-1에서 절대값 증가
최근 등급변화폭 vs 집약도변화율 Spearman,-0.0875,-0.2372,방향 동일; t-1에서 절대값 증가
최근 상향 집약도증감률 중앙값(%),-4.0070,-5.1678,방향 동일; t-1에서 절대값 증가
최근 유지 집약도증감률 중앙값(%),-2.2629,-1.2253,방향 동일; t-1에서 절대값 감소
최근 하향 집약도증감률 중앙값(%),-2.0477,8.0207,방향 변화; t-1에서 절대값 증가


SAME-YEAR 최근 변화 Spearman p-value=0.0652392, n=445
PRIMARY(t-1) 최근 변화 Spearman p-value=5.11663e-07, n=438


## 업종별 추가 참고

GIR 업종명과 세분화 수준은 연도별로 달라질 수 있다. 예를 들어 같은 산업군이 `철강`과 `1차 철강 제조업`처럼 다른 원본 명칭으로 나타날 수 있다. 임의 매핑으로 강제 통합하지 않고, 아래 표는 정확한 원본 업종명의 표본 분포만 참고용으로 제시한다. 업종 결과를 메인 결론의 근거로 사용하지 않는다.

In [7]:
industry_candidates = [col for col in primary.columns if "업종" in str(col)]
industry_col = industry_candidates[0]
industry_reference = (
    primary.dropna(subset=["배출집약도"])
    .groupby(industry_col)
    .agg(기업연도수=("기업키", "size"), 고유기업수=("기업키", "nunique"))
    .sort_values("기업연도수", ascending=False)
)
print("정확한 원본 업종명 기준 상위 표본(추가 참고)")
display(industry_reference.head(20))

정확한 원본 업종명 기준 상위 표본(추가 참고)


,기업연도수,고유기업수
지정업종(목표)/<br />계획업종(할당),,
산업,196,159
석유화학,75,44
전기전자,56,22
1차 철강 제조업,53,19
기초 화학물질 제조업,49,18
"펄프, 종이 및 판지 제조업",36,13
기계,35,11
"시멘트, 석회, 플라스터 및 그 외 제품 제조업",28,10
철강,27,17


### 데이터 결합

KCGS 평가자료를 정규화 기업명으로 GIR 기업-연도 자료와 결합하고, KCGS 종목코드와 성과연도로 DART Revenue를 결합했다. 정규화명+연도 중복과 서로 다른 원명의 정규화 충돌은 KCGS와 GIR 모두 0건이었다. 최신 결산일의 기업·연도·별도/연결 Revenue 충돌도 0건이었다.

PRIMARY(t-1)는 **1,149개 기업연도·270개 고유기업**이며, 배출집약도는 **1,141개 기업연도·270개 고유기업**에서 유효했다.

### 핵심 결과

1. 절대배출량에서는 환경등급이 높을수록 오히려 큰 경향이 나타났다(Spearman ρ=0.3453). A+ 중앙값은 162,990 tCO₂eq, D 중앙값은 34,404 tCO₂eq로 기업 규모 효과가 크다.
2. t-1 기준 배출집약도 중앙값은 A+ 50.8, A 63.5, B+ 89.1, B 136.6, C 172.3, D 186.5 tCO₂eq/매출 10억원이었다. 등급이 높을수록 낮은 경향이 나타났고 전체 Spearman은 ρ=-0.3341, p=3.68e-31이었다.
3. 동일기업의 최근 평가년도 구간에서 등급 상향·유지·하향 그룹의 집약도 증감률 중앙값은 각각 -5.17%, -1.23%, +8.02%였다. 등급 변화폭과 집약도 변화율의 Spearman은 ρ=-0.2372, p=5.12e-7, 유효비교 438건이었다. PRIMARY의 평가년도 2023→2024와 2024→2025 변화는 실제 성과연도 기준으로 각각 2022→2023과 2023→2024 변화에 연결된다.
4. 이 결과는 상관관계이며 인과관계가 아니다. KCGS 환경등급이 온실가스만으로 결정된다는 뜻도 아니다.

### 민감도 분석

same-year와 t-1 모두 환경등급과 절대배출량은 양의 관계, 환경등급과 배출집약도는 음의 관계여서 전체 방향은 같았다. 배출집약도 Spearman은 same-year -0.3180에서 t-1 -0.3341로 절대값이 조금 커졌다.

최근 등급 변화폭과 집약도 변화율의 관계는 same-year ρ=-0.0875, p=0.0652였고 t-1은 ρ=-0.2372, p=5.12e-7이었다. 특히 최근 하향 그룹의 집약도 증감률 중앙값은 same-year -2.05%에서 t-1 +8.02%로 방향이 바뀌었다. 따라서 높은 환경등급과 낮은 배출집약도의 전체 상관 방향은 유지되지만, 최근 동일기업 변화에 관한 결론은 t-1 정렬에서 더 뚜렷해진다.

### 한계

- GIR은 관리/할당 대상 기업 중심이다.
- GIR 법인 경계와 DART 재무제표 경계가 완벽히 일치하지 않을 수 있다. 별도재무제표 우선 선택도 이 차이를 완전히 제거하지 못한다.
- KCGS 환경등급은 온실가스만 평가하는 지표가 아니다.
- 업종 분류체계가 연도별로 변하므로 원본 업종명 기준 참고표를 강한 결론의 근거로 사용하지 않았다.
- 같은 기업이 여러 연도 반복 관측되므로 단순 Spearman p-value의 독립성 가정에 제한이 있다.
- 증감률의 1%·99% 분위수를 확인했지만 극단값을 자동 삭제하지 않았다.
- t-1 정렬과 관측된 상관관계는 인과관계를 의미하지 않는다.

### 실무적 의미

환경성과 관리는 ESG 환경등급 하나만 보는 것이 아니라 다음 항목을 함께 확인해야 한다.

- 절대배출량
- 매출 대비 배출집약도
- 전년 대비 변화
- 평가등급
- 감축목표